### Import dependencies

In [17]:
import openai
import pandas as pd
import tiktoken
from qdrant_client import QdrantClient, models
from qdrant_client.models import VectorParams, Distance, SparseVectorParams, Modifier,PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery

In [2]:
from dotenv import load_dotenv

load_dotenv('../../.env')

True

### Create new collection in Qdrant

In [3]:
qdrant_client = QdrantClient(url='http://localhost:6333')

In [4]:
qdrant_client.create_collection(
    collection_name='amazon-reviews-collection-01',
    vectors_config={
        "text-embedding-3-small": VectorParams(size=1536, distance=Distance.COSINE),
    }
)

True

### Set index

In [5]:
qdrant_client.create_payload_index(
    collection_name='amazon-reviews-collection-01',
    field_name='parent_asin',
    field_schema=PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

### Embedding functions

In [6]:
def get_embedding(text, model='text-embedding-3-small'):
    response = openai.embeddings.create(
        model=model,
        input=text
    )
    return response.data[0].embedding

In [7]:
def get_embeddings_batch(text_list, model='text-embedding-3-small', batch_size=100):
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]
    
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Batch {counter * batch_size} completed of {len(text_list)}")
        counter += 1

    return all_embeddings

#### Read Sampled Data with Amazon items

In [12]:
df_reviews = pd.read_json('../../data/Electronics_recent_2022_2023_has_main_category_ratings_100_sample_1000.jsonl', lines=True)

In [13]:
df_reviews.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,4,Nixplay 10.1 touch screen digital picture frame,I purchased this digital frame on a treasure t...,[],B096DQF21Z,B0BNXXNBB4,AFZUK3MTBIBEDQOPAK3OATUOUKLA,2022-07-29 06:52:30.702,19,True
1,5,Great so far...,"Speedy delivery, great sound and a great warra...",[],B08H1WNYTR,B0C72D4J46,AFANVB6MPHJTCTFOVIEBKLWZ2GVA,2022-07-05 14:40:48.001,0,True
2,5,Set up is not that easy.,Nice looking set but installation instructions...,[],B09XGX1GMJ,B09XGQQ98K,AHACLF2COQQE2V33ZFXQ7THZOJ2Q,2022-09-21 11:26:42.074,0,True
3,2,Waste of money,"Very unhappy with this keyboard, it would slip...",[],B07899MFZ2,B07L5L22ZL,AGYEAZK4OEYF2MSSTGJ5WNJDVZKA,2018-09-29 22:39:47.708,1,True
4,5,Nice,Work great,[],B09JSMNZRG,B09LTX3SQX,AH67BI7JTOFR35HMZYFVOEHM4CPQ,2023-01-11 21:15:08.805,0,True


In [14]:
len(df_reviews)

122840

### Preprocess reviews and count tokens

In [16]:
def preprocess_reviews_data(row):
    return f"{row['title']} {row['text']}"

In [26]:
def count_tokens(row):
    encoding = tiktoken.encoding_for_model("text-embedding-3-small")
    return len(encoding.encode(row['preprocessed_data']))

In [27]:
df_reviews['preprocessed_data'] = df_reviews.apply(preprocess_reviews_data, axis=1)
df_reviews['token_count'] = df_reviews.apply(count_tokens, axis=1)

In [28]:
df_reviews.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,preprocessed_data,token_count
0,4,Nixplay 10.1 touch screen digital picture frame,I purchased this digital frame on a treasure t...,[],B096DQF21Z,B0BNXXNBB4,AFZUK3MTBIBEDQOPAK3OATUOUKLA,2022-07-29 06:52:30.702,19,True,Nixplay 10.1 touch screen digital picture fram...,255
1,5,Great so far...,"Speedy delivery, great sound and a great warra...",[],B08H1WNYTR,B0C72D4J46,AFANVB6MPHJTCTFOVIEBKLWZ2GVA,2022-07-05 14:40:48.001,0,True,"Great so far... Speedy delivery, great sound a...",27
2,5,Set up is not that easy.,Nice looking set but installation instructions...,[],B09XGX1GMJ,B09XGQQ98K,AHACLF2COQQE2V33ZFXQ7THZOJ2Q,2022-09-21 11:26:42.074,0,True,Set up is not that easy. Nice looking set but ...,18
3,2,Waste of money,"Very unhappy with this keyboard, it would slip...",[],B07899MFZ2,B07L5L22ZL,AGYEAZK4OEYF2MSSTGJ5WNJDVZKA,2018-09-29 22:39:47.708,1,True,Waste of money Very unhappy with this keyboard...,66
4,5,Nice,Work great,[],B09JSMNZRG,B09LTX3SQX,AH67BI7JTOFR35HMZYFVOEHM4CPQ,2023-01-11 21:15:08.805,0,True,Nice Work great,3


In [29]:
len(df_reviews)

122840

In [ ]:
df_reviews = df_reviews[df_reviews['token_count'] < 8192] #this is the max number of tokens for the embedding model to process
len(df_reviews)

122840

In [31]:
total_tokens = df_reviews['token_count'].sum()

In [ ]:
total_tokens #this number times the price per 1M token gives the total cost of the embeddings

np.int64(6506026)

### Add data to Qdrant collection

In [33]:
relevant_columns = ['preprocessed_data', 'parent_asin']
df_data_to_embed = df_reviews[relevant_columns]

In [35]:
data_to_embed = df_data_to_embed.to_dict(orient='records')

In [36]:
text_to_embed = [item['preprocessed_data'] for item in data_to_embed]

In [37]:
embeddings = get_embeddings_batch(text_to_embed)

Batch 100 completed of 122840
Batch 200 completed of 122840
Batch 300 completed of 122840
Batch 400 completed of 122840
Batch 500 completed of 122840
Batch 600 completed of 122840
Batch 700 completed of 122840
Batch 800 completed of 122840
Batch 900 completed of 122840
Batch 1000 completed of 122840
Batch 1100 completed of 122840
Batch 1200 completed of 122840
Batch 1300 completed of 122840
Batch 1400 completed of 122840
Batch 1500 completed of 122840
Batch 1600 completed of 122840
Batch 1700 completed of 122840
Batch 1800 completed of 122840
Batch 1900 completed of 122840
Batch 2000 completed of 122840
Batch 2100 completed of 122840
Batch 2200 completed of 122840
Batch 2300 completed of 122840
Batch 2400 completed of 122840
Batch 2500 completed of 122840
Batch 2600 completed of 122840
Batch 2700 completed of 122840
Batch 2800 completed of 122840
Batch 2900 completed of 122840
Batch 3000 completed of 122840
Batch 3100 completed of 122840
Batch 3200 completed of 122840
Batch 3300 comple

In [39]:
len(embeddings)

122840

### Add vectors to Qdrant

In [40]:
point_structs = []
i = 1
for embedding, data in zip(embeddings, data_to_embed):
    point_structs.append(
        PointStruct(
            id=i,
            vector={
                "text-embedding-3-small": embedding
            },
            payload=data
        )
    )
    i += 1

In [41]:
batch_size = 500
counter = 1
for i in range(0, len(point_structs), batch_size):
    batch = point_structs[i:i + batch_size]
    qdrant_client.upsert(
        collection_name='amazon-reviews-collection-01',
        points=batch,
        wait=True
    )
    print(f"Batch {counter * batch_size} completed of {len(point_structs)}")
    counter += 1

Batch 500 completed of 122840
Batch 1000 completed of 122840
Batch 1500 completed of 122840
Batch 2000 completed of 122840
Batch 2500 completed of 122840
Batch 3000 completed of 122840
Batch 3500 completed of 122840
Batch 4000 completed of 122840
Batch 4500 completed of 122840
Batch 5000 completed of 122840
Batch 5500 completed of 122840
Batch 6000 completed of 122840
Batch 6500 completed of 122840
Batch 7000 completed of 122840
Batch 7500 completed of 122840
Batch 8000 completed of 122840
Batch 8500 completed of 122840
Batch 9000 completed of 122840
Batch 9500 completed of 122840
Batch 10000 completed of 122840
Batch 10500 completed of 122840
Batch 11000 completed of 122840
Batch 11500 completed of 122840
Batch 12000 completed of 122840
Batch 12500 completed of 122840
Batch 13000 completed of 122840
Batch 13500 completed of 122840
Batch 14000 completed of 122840
Batch 14500 completed of 122840
Batch 15000 completed of 122840
Batch 15500 completed of 122840
Batch 16000 completed of 122

### Function to run search against reviews on a prefiltered set of product IDs

In [42]:
def retrieve_prefiltered_reviews_data(query, parent_asins, collection_name='amazon-reviews-collection-01', k=5):
    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name=collection_name,
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                filter=models.Filter(
                    must=[
                        models.FieldCondition(
                            key="parent_asin",
                            match=models.MatchAny(any=parent_asins)
                        )
                    ]
                ),
                limit=20
            )
        ],
        query=FusionQuery(fusion='rrf'),
        limit=k
    )

    return results

In [48]:
reviews = retrieve_prefiltered_reviews_data(query='bad quality', parent_asins=['B09Q5TNDHY'])

In [49]:
reviews.points

[ScoredPoint(id=62005, version=127, score=0.5, payload={'preprocessed_data': 'Its a total waste and the screen came as if it was used before The screen unlike the picture is so small and when i took it out of the box it looks like the watch was used before its not new<br />wouldn’t recommend', 'parent_asin': 'B09Q5TNDHY'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=47673, version=98, score=0.33333334, payload={'preprocessed_data': 'Not worth the money Watch stopped working after four months.', 'parent_asin': 'B09Q5TNDHY'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=62009, version=127, score=0.25, payload={'preprocessed_data': "It's junk didn't last more then a week Screen totally blanket out. Don't waste your money", 'parent_asin': 'B09Q5TNDHY'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=80491, version=163, score=0.2, payload={'preprocessed_data': "Its stopped working It's not that durable", 'parent_asin': 'B09Q5TNDHY'}, v

### Define the reviews retrieval tool

In [54]:
def get_embedding(text, model='text-embedding-3-small'):
    response = openai.embeddings.create(
        model=model,
        input=text,
    )
    
    return response.data[0].embedding

def retrieve_prefiltered_reviews_data(query, parent_asins, collection_name='amazon-reviews-collection-01', k=5):
    query_embedding = get_embedding(query)
    qdrant_client = QdrantClient(url="http://localhost:6333")

    results = qdrant_client.query_points(
        collection_name=collection_name,
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                filter=models.Filter(
                    must=[
                        models.FieldCondition(
                            key="parent_asin",
                            match=models.MatchAny(any=parent_asins)
                        )
                    ]
                ),
                limit=20
            )
        ],
        query=FusionQuery(fusion='rrf'),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context_texts = []
    similarity_scores = []

    for result in results.points:
        retrieved_context_ids.append(result.payload['parent_asin'])
        retrieved_context_texts.append(result.payload['preprocessed_data'])
        similarity_scores.append(result.score)
    return {
        'retrieved_context_ids': retrieved_context_ids,
        'retrieved_context_texts': retrieved_context_texts,
        'similarity_scores': similarity_scores
    }

def process_context(retrieve_context):
    formatted_context = ''

    for id, chunk in zip(retrieve_context['retrieved_context_ids'], retrieve_context['retrieved_context_texts']):
        formatted_context += f"- Product ID: {id}, Product Review: {chunk}\n"

    return formatted_context

def get_formatted_reviews_context(query: str, parent_asins: list[str], top_k: int = 5) -> str:
    """
    Get the top_k reviews matching a query for a list of prefiltered items.
    Args:
        query: The query to get the top k reviews for
        parent_asins: The list of item IDs to prefilter for before running the query
        top_k: The number of reviews to retrieve, this should be at least 20 if multiple items are being filtered
    Returns:
        A string of the top_k context chunks with IDs and average ratings prepeding each chunk, each representing an inventory item for a given query
    """

    retrieved_context = retrieve_prefiltered_reviews_data(query, parent_asins, k=top_k)

    formatted_context = process_context(retrieved_context)

    return formatted_context

In [55]:
reviews = get_formatted_reviews_context(query='bad quality', parent_asins=['B09Q5TNDHY'])

In [57]:
print(reviews)

- Product ID: B09Q5TNDHY, Product Review: Its a total waste and the screen came as if it was used before The screen unlike the picture is so small and when i took it out of the box it looks like the watch was used before its not new<br />wouldn’t recommend
- Product ID: B09Q5TNDHY, Product Review: Not worth the money Watch stopped working after four months.
- Product ID: B09Q5TNDHY, Product Review: It's junk didn't last more then a week Screen totally blanket out. Don't waste your money
- Product ID: B09Q5TNDHY, Product Review: Its stopped working It's not that durable
- Product ID: B09Q5TNDHY, Product Review: Didn't work I don't know if this product was defective or if it's a poor design or whatever. I'll be sending it back right away. I got it fully charged set up ,<br />went swimming, it worked during the swimming. But afterwards it cease to function and begin to act strangely vibrating almost continually.

